In [ ]:
# install libraries

%pip install pandas
%pip install scikit-learn
%pip install numpy
%pip install torch
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.5 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# SYNTHETIC MIMIC-IV GENERATOR
# NOTEBOOK 1
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

np.random.seed(42)
random.seed(42)

# ==========================================================
# CONFIG
# ==========================================================

N_PATIENTS = 10000
MAX_ADMISSIONS = 4

SAVE_PATH = "/content/drive/MyDrive/synthetic_mimic"

os.makedirs(SAVE_PATH, exist_ok=True)

print("Generating synthetic dataset...")

# ==========================================================
# PATIENTS
# ==========================================================

patients = pd.DataFrame({
    "subject_id": np.arange(100000, 100000 + N_PATIENTS),
    "anchor_age": np.random.randint(18,95,N_PATIENTS),
    "gender": np.random.choice(
        ["M","F","Male","Female","Unknown"],
        N_PATIENTS,
        p=[0.42,0.42,0.06,0.06,0.04]
    ),
    "race": np.random.choice(
        [
            "WHITE",
            "BLACK",
            "ASIAN",
            "HISPANIC",
            "OTHER",
            None
        ],
        N_PATIENTS
    )
})

# duplicates
patients = pd.concat([
    patients,
    patients.sample(200)
])

# ==========================================================
# ADMISSIONS
# ==========================================================

admissions_rows = []

hadm_id = 200000

for pid in patients["subject_id"].unique():

    n_adm = np.random.randint(
        1,
        MAX_ADMISSIONS + 1
    )

    for _ in range(n_adm):

        admit = (
            datetime(2018,1,1)
            +
            timedelta(
                days=np.random.randint(0,2200)
            )
        )

        los = np.random.randint(1,20)

        discharge = admit + timedelta(days=los)

        admissions_rows.append([
            hadm_id,
            pid,
            admit,
            discharge,
            los,
            np.random.choice([
                "EMERGENCY",
                "URGENT",
                "ELECTIVE",
                "OBSERVATION"
            ]),
            np.random.choice([
                "HOME",
                "REHAB",
                "SNF",
                "EXPIRED"
            ])
        ])

        hadm_id += 1

admissions = pd.DataFrame(
    admissions_rows,
    columns=[
        "hadm_id",
        "subject_id",
        "admittime",
        "dischtime",
        "los_days",
        "admission_type",
        "discharge_location"
    ]
)

# ==========================================================
# ICU STAYS
# ==========================================================

icu = admissions[
    ["hadm_id","subject_id"]
].copy()

icu["icu_los"] = np.random.gamma(
    shape=2,
    scale=2,
    size=len(icu)
)

icu["icu_los"] = icu["icu_los"].round(2)

icu["sofa"] = np.random.randint(
    0,
    18,
    len(icu)
)

# ==========================================================
# DIAGNOSES
# ==========================================================

codes = [
    "I10",
    "E11",
    "J18",
    "N17",
    "I50",
    "A41",
    "K92"
]

diag_rows = []

for _, row in admissions.iterrows():

    n_dx = np.random.randint(1,6)

    for seq in range(n_dx):

        diag_rows.append([
            row["subject_id"],
            row["hadm_id"],
            seq + 1,
            random.choice(codes)
        ])

diagnoses = pd.DataFrame(
    diag_rows,
    columns=[
        "subject_id",
        "hadm_id",
        "seq_num",
        "icd_code"
    ]
)

# ==========================================================
# PROCEDURES
# ==========================================================

procedure_names = [
    "XRAY",
    "CT_SCAN",
    "MRI",
    "INTUBATION",
    "DIALYSIS",
    "SURGERY"
]

proc_rows = []

for _, row in admissions.iterrows():

    n_proc = np.random.randint(1,5)

    for _ in range(n_proc):

        proc_rows.append([
            row["hadm_id"],
            random.choice(
                procedure_names
            )
        ])

procedures = pd.DataFrame(
    proc_rows,
    columns=[
        "hadm_id",
        "procedure_name"
    ]
)

# ==========================================================
# MEDICATIONS
# ==========================================================

meds = [
    "Metformin",
    "Insulin",
    "Aspirin",
    "Vancomycin",
    "Lisinopril",
    "Furosemide"
]

med_rows = []

for _, row in admissions.iterrows():

    n_med = np.random.randint(1,8)

    for _ in range(n_med):

        med_rows.append([
            row["hadm_id"],
            random.choice(meds),
            np.random.choice([
                "5mg",
                "10mg",
                "20mg",
                "500mg",
                "1g",
                "UNKNOWN"
            ])
        ])

medications = pd.DataFrame(
    med_rows,
    columns=[
        "hadm_id",
        "drug",
        "dose"
    ]
)

# ==========================================================
# LABS
# ==========================================================

lab_names = [
    "WBC",
    "Hemoglobin",
    "Platelets",
    "Creatinine",
    "Glucose"
]

lab_rows = []

for _, row in admissions.iterrows():

    n_labs = np.random.randint(5,25)

    for _ in range(n_labs):

        value = abs(
            np.random.normal(
                100,
                30
            )
        )

        if np.random.rand() < 0.01:
            value *= 10

        lab_rows.append([
            row["hadm_id"],
            random.choice(
                lab_names
            ),
            round(value,2)
        ])

labs = pd.DataFrame(
    lab_rows,
    columns=[
        "hadm_id",
        "lab_name",
        "lab_value"
    ]
)

# missing values
labs.loc[
    labs.sample(
        frac=0.05
    ).index,
    "lab_value"
] = np.nan

# ==========================================================
# COMORBIDITIES
# ==========================================================

conditions = [
    "Diabetes",
    "Hypertension",
    "CHF",
    "COPD",
    "CKD"
]

comorb_rows = []

for pid in patients[
    "subject_id"
].unique():

    n = np.random.randint(
        1,
        4
    )

    selected = np.random.choice(
        conditions,
        n,
        replace=False
    )

    for c in selected:

        comorb_rows.append([
            pid,
            c
        ])

comorbidities = pd.DataFrame(
    comorb_rows,
    columns=[
        "subject_id",
        "condition"
    ]
)

# ==========================================================
# OUTCOMES
# ==========================================================

outcomes = admissions[
    ["hadm_id","los_days"]
].copy()

outcomes["mortality"] = (
    np.random.rand(
        len(outcomes)
    )
    <
    (
        outcomes["los_days"]/100
    )
).astype(int)

outcomes["mortality_90d"] = (
    np.random.rand(
        len(outcomes)
    )
    <
    (
        outcomes["los_days"]/120
    )
).astype(int)

outcomes["readmit_30d"] = (
    np.random.rand(
        len(outcomes)
    )
    < 0.15
).astype(int)

outcomes["sepsis"] = (
    np.random.rand(
        len(outcomes)
    )
    < 0.12
).astype(int)

outcomes["ventilation"] = (
    np.random.rand(
        len(outcomes)
    )
    < 0.08
).astype(int)

# ==========================================================
# SAVE FILES
# ==========================================================

patients.to_csv(
    f"{SAVE_PATH}/patients.csv",
    index=False
)

admissions.to_csv(
    f"{SAVE_PATH}/admissions.csv",
    index=False
)

icu.to_csv(
    f"{SAVE_PATH}/icu_stays.csv",
    index=False
)

diagnoses.to_csv(
    f"{SAVE_PATH}/diagnoses.csv",
    index=False
)

procedures.to_csv(
    f"{SAVE_PATH}/procedures.csv",
    index=False
)

medications.to_csv(
    f"{SAVE_PATH}/medications.csv",
    index=False
)

labs.to_csv(
    f"{SAVE_PATH}/labs.csv",
    index=False
)

comorbidities.to_csv(
    f"{SAVE_PATH}/comorbidities.csv",
    index=False
)

outcomes.to_csv(
    f"{SAVE_PATH}/outcomes.csv",
    index=False
)

print("\nDATASET CREATED\n")

for name, df in {
    "patients": patients,
    "admissions": admissions,
    "icu": icu,
    "diagnoses": diagnoses,
    "procedures": procedures,
    "medications": medications,
    "labs": labs,
    "comorbidities": comorbidities,
    "outcomes": outcomes
}.items():

    print(
        f"{name}: {df.shape}"
    )

print(
    "\nSaved to:",
    SAVE_PATH
)

Mounted at /content/drive
Generating synthetic dataset...

DATASET CREATED

patients: (10200, 4)
admissions: (25059, 7)
icu: (25059, 4)
diagnoses: (74810, 4)
procedures: (62542, 2)
medications: (100576, 3)
labs: (363744, 3)
comorbidities: (19961, 2)
outcomes: (25059, 7)

Saved to: /content/drive/MyDrive/synthetic_mimic
